In [ ]:
# CEFR Level Classification Model - Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("="*60)
print("CEFR TEXT CLASSIFICATION MODEL")
print("="*60)
print("\nLibraries loaded successfully!")

CEFR TEXT CLASSIFICATION MODEL

Libraries loaded successfully!


In [ ]:
# Load Dataset
print("="*60)
print("LOADING DATASET")
print("="*60)

df = pd.read_csv('/kaggle/input/datasets/ahmedsameh72/cefr-last/cefr_large_dataset-2.csv')

print(f"\nDataset loaded!")
print(f"Total samples: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()

LOADING DATASET

Dataset loaded!
Total samples: 151816
Columns: ['text', 'label', 'source']

First few rows:


,text,label,source
0,I just had my third interview with a company. ...,B2,reddit
1,Just wondering what made you decide you needed...,B1,reddit
2,Some background: I'm from California and my gi...,B2,reddit
3,Hey guys:\n\nHere's a rough draft I'm submitti...,C2,reddit
4,"**Section A:** *It's BACON!*\n\nAll ministers,...",B2,reddit


In [ ]:
# Data Preprocessing - Step 1: Basic Cleaning
print("="*60)
print("STEP 1: BASIC DATA CLEANING")
print("="*60)

print(f"\nOriginal dataset size: {len(df)}")

# Create a copy for processing
df_clean = df.copy()

# Remove any null values
print(f"Null values per column:\n{df_clean.isnull().sum()}")
df_clean = df_clean.dropna()
print(f"After removing nulls: {len(df_clean)}")

# Remove duplicates based on text
duplicates = df_clean['text'].duplicated().sum()
df_clean = df_clean.drop_duplicates(subset=['text'], keep='first')
print(f"Duplicates removed: {duplicates}")
print(f"After removing duplicates: {len(df_clean)}")

# Strip whitespace
df_clean['text'] = df_clean['text'].astype(str).str.strip()

# Calculate text statistics
df_clean['text_length'] = df_clean['text'].apply(len)
df_clean['word_count'] = df_clean['text'].apply(lambda x: len(x.split()))

print(f"\nFinal cleaned dataset: {len(df_clean)} samples")
print("="*60)

STEP 1: BASIC DATA CLEANING

Original dataset size: 151816
Null values per column:
text      0
label     0
source    0
dtype: int64
After removing nulls: 151816
Duplicates removed: 0
After removing duplicates: 151816

Final cleaned dataset: 151816 samples


In [ ]:
# Data Preprocessing - Step 2: Analyze Class Distribution
print("="*60)
print("STEP 2: CLASS DISTRIBUTION ANALYSIS")
print("="*60)

print("\nCEFR Level Distribution:")
print("-"*60)
label_counts = df_clean['label'].value_counts().sort_index()
label_percentages = (df_clean['label'].value_counts(normalize=True).sort_index() * 100)

distribution_df = pd.DataFrame({
    'Count': label_counts,
    'Percentage': label_percentages.round(2)
})
print(distribution_df)

print("\n" + "="*60)
print("⚠️  CLASS IMBALANCE DETECTED!")
print("="*60)
print("This will be handled using class weights during training.")
print("="*60)

STEP 2: CLASS DISTRIBUTION ANALYSIS

CEFR Level Distribution:
------------------------------------------------------------
       Count  Percentage
label                   
A1       167        0.11
A2     40000       26.35
B1     40000       26.35
B2     40000       26.35
C1     21286       14.02
C2     10363        6.83

⚠️  CLASS IMBALANCE DETECTED!
This will be handled using class weights during training.


In [ ]:
# Data Preprocessing - Step 3: Filter Outliers (Optional)
print("="*60)
print("STEP 3: OUTLIER ANALYSIS")
print("="*60)

print(f"\nCurrent dataset size: {len(df_clean)}")
print(f"\nWord count statistics:")
print(df_clean['word_count'].describe())

# Analyze outliers
print("\n" + "-"*60)
print("SHORT TEXTS:")
for threshold in [5, 10, 20]:
    short_count = len(df_clean[df_clean['word_count'] < threshold])
    print(f"  < {threshold} words: {short_count} ({short_count/len(df_clean)*100:.2f}%)")

print("\nLONG TEXTS:")
for threshold in [500, 750, 1000]:
    long_count = len(df_clean[df_clean['word_count'] > threshold])
    print(f"  > {threshold} words: {long_count} ({long_count/len(df_clean)*100:.2f}%)")

print("\n" + "-"*60)
print("FILTERING OPTIONS:")
print("-"*60)
print("1. Keep all data (no filtering)")
print("2. Remove very short texts (< 10 words)")
print("3. Remove very short texts (< 20 words)")
print("4. Remove very long texts (> 1000 words)")
print("5. Remove both short (< 20) and long (> 1000)")
print("-"*60)

# Default: Apply minimal filtering (< 10 words, very short texts)
# You can change this based on your preference
print("\n⚙️  Applying default filter: removing texts < 10 words")
min_words = 10
df_processed = df_clean[df_clean['word_count'] >= min_words].copy()
print(f"Samples removed: {len(df_clean) - len(df_processed)}")
print(f"Final dataset size: {len(df_processed)}")

print("\n💡 To change filtering, modify 'min_words' variable above and re-run")
print("="*60)

STEP 3: OUTLIER ANALYSIS

Current dataset size: 151816

Word count statistics:
count    151816.000000
mean        127.495639
std         127.092042
min           1.000000
25%          42.000000
50%          84.000000
75%         165.000000
max        1322.000000
Name: word_count, dtype: float64

------------------------------------------------------------
SHORT TEXTS:
  < 5 words: 25 (0.02%)
  < 10 words: 1686 (1.11%)
  < 20 words: 10617 (6.99%)

LONG TEXTS:
  > 500 words: 3842 (2.53%)
  > 750 words: 235 (0.15%)
  > 1000 words: 19 (0.01%)

------------------------------------------------------------
FILTERING OPTIONS:
------------------------------------------------------------
1. Keep all data (no filtering)
2. Remove very short texts (< 10 words)
3. Remove very short texts (< 20 words)
4. Remove very long texts (> 1000 words)
5. Remove both short (< 20) and long (> 1000)
------------------------------------------------------------

⚙️  Applying default filter: removing texts < 10 wor

In [ ]:
# Data Preprocessing - Step 4: Prepare Features and Labels
print("="*60)
print("STEP 4: PREPARE FEATURES AND LABELS")
print("="*60)

# Extract text and labels
X = df_processed['text'].values
y = df_processed['label'].values

print(f"\nFeatures (X): {X.shape}")
print(f"Labels (y): {y.shape}")
print(f"\nUnique labels: {np.unique(y)}")
print(f"Label distribution:\n{pd.Series(y).value_counts().sort_index()}")

# Encode labels to integers
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"\nLabel encoding mapping:")
for i, label in enumerate(label_encoder.classes_):
    print(f"  {label} -> {i}")

print(f"\nEncoded labels shape: {y_encoded.shape}")
print("="*60)

STEP 4: PREPARE FEATURES AND LABELS

Features (X): (150130,)
Labels (y): (150130,)

Unique labels: ['A1' 'A2' 'B1' 'B2' 'C1' 'C2']
Label distribution:
A1       79
A2    39379
B1    39430
B2    39738
C1    21153
C2    10351
Name: count, dtype: int64

Label encoding mapping:
  A1 -> 0
  A2 -> 1
  B1 -> 2
  B2 -> 3
  C1 -> 4
  C2 -> 5

Encoded labels shape: (150130,)


In [ ]:
# Data Preprocessing - Step 5: Train-Test Split
print("="*60)
print("STEP 5: TRAIN-TEST-VALIDATION SPLIT")
print("="*60)

# First split: separate test set (15%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y_encoded,
    test_size=0.15,
    random_state=RANDOM_STATE,
    stratify=y_encoded
)

# Second split: separate validation set (15% of remaining = ~12.75% of total)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.15,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

print(f"\nDataset splits:")
print(f"  Training set:   {len(X_train)} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"  Validation set: {len(X_val)} samples ({len(X_val)/len(X)*100:.1f}%)")
print(f"  Test set:       {len(X_test)} samples ({len(X_test)/len(X)*100:.1f}%)")

print(f"\n\nTraining set label distribution:")
train_dist = pd.Series(y_train).value_counts().sort_index()
for idx, count in train_dist.items():
    label_name = label_encoder.classes_[idx]
    print(f"  {label_name}: {count} ({count/len(y_train)*100:.1f}%)")

print(f"\n\nValidation set label distribution:")
val_dist = pd.Series(y_val).value_counts().sort_index()
for idx, count in val_dist.items():
    label_name = label_encoder.classes_[idx]
    print(f"  {label_name}: {count} ({count/len(y_val)*100:.1f}%)")

print(f"\n\nTest set label distribution:")
test_dist = pd.Series(y_test).value_counts().sort_index()
for idx, count in test_dist.items():
    label_name = label_encoder.classes_[idx]
    print(f"  {label_name}: {count} ({count/len(y_test)*100:.1f}%)")

print("="*60)

STEP 5: TRAIN-TEST-VALIDATION SPLIT

Dataset splits:
  Training set:   108468 samples (72.2%)
  Validation set: 19142 samples (12.8%)
  Test set:       22520 samples (15.0%)


Training set label distribution:
  A1: 57 (0.1%)
  A2: 28451 (26.2%)
  B1: 28489 (26.3%)
  B2: 28710 (26.5%)
  C1: 15283 (14.1%)
  C2: 7478 (6.9%)


Validation set label distribution:
  A1: 10 (0.1%)
  A2: 5021 (26.2%)
  B1: 5027 (26.3%)
  B2: 5067 (26.5%)
  C1: 2697 (14.1%)
  C2: 1320 (6.9%)


Test set label distribution:
  A1: 12 (0.1%)
  A2: 5907 (26.2%)
  B1: 5914 (26.3%)
  B2: 5961 (26.5%)
  C1: 3173 (14.1%)
  C2: 1553 (6.9%)


In [ ]:
# Data Preprocessing - Step 6: Calculate Class Weights
print("="*60)
print("STEP 6: COMPUTE CLASS WEIGHTS FOR IMBALANCE")
print("="*60)

# Compute class weights (inverse frequency)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weight_dict = dict(enumerate(class_weights))

print("\nClass Weights (higher weight = rarer class):")
print("-"*60)
for class_idx, weight in class_weight_dict.items():
    label_name = label_encoder.classes_[class_idx]
    count = (y_train == class_idx).sum()
    percentage = (count / len(y_train)) * 100
    print(f"  {label_name} (Class {class_idx}): Weight = {weight:.2f} | Count = {count} ({percentage:.2f}%)")

print("\n" + "="*60)
print("✅ Class weights will give higher importance to minority classes")
print("   (e.g., A1 with ~0.1% will have much higher weight)")
print("="*60)

STEP 6: COMPUTE CLASS WEIGHTS FOR IMBALANCE

Class Weights (higher weight = rarer class):
------------------------------------------------------------
  A1 (Class 0): Weight = 317.16 | Count = 57 (0.05%)
  A2 (Class 1): Weight = 0.64 | Count = 28451 (26.23%)
  B1 (Class 2): Weight = 0.63 | Count = 28489 (26.26%)
  B2 (Class 3): Weight = 0.63 | Count = 28710 (26.47%)
  C1 (Class 4): Weight = 1.18 | Count = 15283 (14.09%)
  C2 (Class 5): Weight = 2.42 | Count = 7478 (6.89%)

✅ Class weights will give higher importance to minority classes
   (e.g., A1 with ~0.1% will have much higher weight)


In [ ]:
# Save Preprocessed Data
print("="*60)
print("SAVING PREPROCESSED DATA")
print("="*60)

# Create a processed dataframe
df_final = df_processed[['text', 'label', 'source']].copy()

# Save to CSV
output_file = 'cefr_dataset_processed.csv'
df_final.to_csv(output_file, index=False)
print(f"\n✅ Preprocessed data saved to: {output_file}")
print(f"   Total samples: {len(df_final)}")

# Save label encoder
import pickle
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)
print(f"✅ Label encoder saved to: label_encoder.pkl")

# Save split data for model training
np.save('X_train.npy', X_train)
np.save('X_val.npy', X_val)
np.save('X_test.npy', X_test)
np.save('y_train.npy', y_train)
np.save('y_val.npy', y_val)
np.save('y_test.npy', y_test)
print(f"✅ Train/Val/Test splits saved")

# Save class weights
with open('class_weights.pkl', 'wb') as f:
    pickle.dump(class_weight_dict, f)
print(f"✅ Class weights saved to: class_weights.pkl")

print("\n" + "="*60)
print("PREPROCESSING COMPLETE!")
print("="*60)

SAVING PREPROCESSED DATA

✅ Preprocessed data saved to: cefr_dataset_processed.csv
   Total samples: 150130
✅ Label encoder saved to: label_encoder.pkl
✅ Train/Val/Test splits saved
✅ Class weights saved to: class_weights.pkl

PREPROCESSING COMPLETE!


In [ ]:
# Install required libraries (run once)
# !pip install transformers torch scikit-learn datasets accelerate -q
print("✅ Make sure transformers and torch are installed!")
print("   Run: pip install transformers torch scikit-learn datasets accelerate")

✅ Make sure transformers and torch are installed!
   Run: pip install transformers torch scikit-learn datasets accelerate


In [ ]:
# Import Transformer Libraries
print("="*60)
print("IMPORTING TRANSFORMER LIBRARIES")
print("="*60)

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import time

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n🖥️  Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("   ⚠️  No GPU detected. Training will be slower on CPU.")

print("="*60)

IMPORTING TRANSFORMER LIBRARIES

🖥️  Device: cuda
   GPU: Tesla T4
   Memory: 14.56 GB


In [ ]:
# Create Dataset Class
class CEFRDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

print("✅ Dataset class created!")

✅ Dataset class created!


In [ ]:
# Helper Functions for Training and Evaluation
def compute_metrics(pred):
    """Compute accuracy and F1 scores"""
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average='macro')
    f1_weighted = f1_score(labels, preds, average='weighted')

    return {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted
    }

def evaluate_model(trainer, dataset, label_encoder, dataset_name="Test"):
    """Detailed evaluation with classification report and confusion matrix"""
    print(f"\n{'='*60}")
    print(f"EVALUATING ON {dataset_name.upper()} SET")
    print('='*60)

    predictions = trainer.predict(dataset)
    pred_labels = predictions.predictions.argmax(-1)
    true_labels = predictions.label_ids

    # Accuracy and F1
    acc = accuracy_score(true_labels, pred_labels)
    f1_macro = f1_score(true_labels, pred_labels, average='macro')
    f1_weighted = f1_score(true_labels, pred_labels, average='weighted')

    print(f"\n📊 Overall Metrics:")
    print(f"   Accuracy:    {acc:.4f} ({acc*100:.2f}%)")
    print(f"   F1 Macro:    {f1_macro:.4f}")
    print(f"   F1 Weighted: {f1_weighted:.4f}")

    # Classification Report
    print(f"\n📋 Per-Class Performance:")
    print("-"*60)
    class_names = label_encoder.classes_
    print(classification_report(true_labels, pred_labels, target_names=class_names, digits=4))

    # Confusion Matrix
    print(f"\n🔢 Confusion Matrix:")
    print("-"*60)
    cm = confusion_matrix(true_labels, pred_labels)
    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    print(cm_df)

    return {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'predictions': pred_labels,
        'true_labels': true_labels
    }

print("✅ Helper functions created!")

✅ Helper functions created!


---
# 🤖 MODEL 1: DistilBERT

Fast and efficient transformer model with excellent performance.
- **Model**: `distilbert-base-uncased`
- **Parameters**: 66M
- **Training Time**: ~30-45 minutes on GPU

---

In [ ]:
# MODEL 1: Load DistilBERT Model and Tokenizer
print("="*60)
print("MODEL 1: DISTILBERT - LOADING")
print("="*60)

distilbert_model_name = "distilbert-base-uncased"
num_labels = len(label_encoder.classes_)

print(f"\nLoading tokenizer: {distilbert_model_name}")
distilbert_tokenizer = AutoTokenizer.from_pretrained(distilbert_model_name)

print(f"Loading model with {num_labels} labels")
distilbert_model = AutoModelForSequenceClassification.from_pretrained(
    distilbert_model_name,
    num_labels=num_labels
).to(device)

print(f"\n✅ DistilBERT loaded successfully!")
print(f"   Parameters: {sum(p.numel() for p in distilbert_model.parameters()) / 1e6:.1f}M")
print("="*60)

MODEL 1: DISTILBERT - LOADING

Loading tokenizer: distilbert-base-uncased


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Loading model with 6 labels


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



✅ DistilBERT loaded successfully!
   Parameters: 67.0M


In [ ]:
# MODEL 1: Prepare DistilBERT Datasets
print("="*60)
print("PREPARING DISTILBERT DATASETS")
print("="*60)

distilbert_train_dataset = CEFRDataset(X_train, y_train, distilbert_tokenizer, max_length=512)
distilbert_val_dataset = CEFRDataset(X_val, y_val, distilbert_tokenizer, max_length=512)
distilbert_test_dataset = CEFRDataset(X_test, y_test, distilbert_tokenizer, max_length=512)

print(f"\n✅ Datasets created:")
print(f"   Training:   {len(distilbert_train_dataset)} samples")
print(f"   Validation: {len(distilbert_val_dataset)} samples")
print(f"   Test:       {len(distilbert_test_dataset)} samples")
print("="*60)

PREPARING DISTILBERT DATASETS

✅ Datasets created:
   Training:   108468 samples
   Validation: 19142 samples
   Test:       22520 samples


In [ ]:
# MODEL 1: Configure DistilBERT Training
print("="*60)
print("CONFIGURING DISTILBERT TRAINING")
print("="*60)

# Convert class weights to tensor
class_weights_tensor = torch.tensor(list(class_weight_dict.values()), dtype=torch.float).to(device)

# Custom Trainer with Class Weights (Updated for latest transformers)
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        # Apply class weights to loss
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

# Training arguments
distilbert_training_args = TrainingArguments(
    output_dir='./distilbert_model',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=500,
    logging_dir='./logs/distilbert',
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    save_total_limit=2,
    seed=RANDOM_STATE,
    fp16=torch.cuda.is_available(),  # Mixed precision training if GPU available
)

# Initialize Trainer
distilbert_trainer = WeightedTrainer(
    model=distilbert_model,
    args=distilbert_training_args,
    train_dataset=distilbert_train_dataset,
    eval_dataset=distilbert_val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("\n✅ DistilBERT Trainer configured!")
print(f"   Epochs: {distilbert_training_args.num_train_epochs}")
print(f"   Batch Size: {distilbert_training_args.per_device_train_batch_size}")
print(f"   Learning Rate: {distilbert_training_args.learning_rate}")
print(f"   Class Weights: Applied ✓")
print("="*60)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


CONFIGURING DISTILBERT TRAINING

✅ DistilBERT Trainer configured!
   Epochs: 4
   Batch Size: 16
   Learning Rate: 2e-05
   Class Weights: Applied ✓


In [ ]:
# MODEL 1: Train DistilBERT
print("="*60)
print("TRAINING DISTILBERT MODEL")
print("="*60)
print("\n🚀 Starting training... This may take 30-45 minutes on GPU\n")

start_time = time.time()

# Train
distilbert_trainer.train()

training_time = time.time() - start_time

print("\n" + "="*60)
print("✅ DISTILBERT TRAINING COMPLETE!")
print("="*60)
print(f"⏱️  Training time: {training_time/60:.2f} minutes")
print("="*60)

TRAINING DISTILBERT MODEL

🚀 Starting training... This may take 30-45 minutes on GPU



Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,0.142449,0.215585,0.953714,0.876095,0.953609
2,0.135271,0.185657,0.955438,0.885202,0.955514
3,0.070285,0.151038,0.959513,0.893534,0.959695
4,0.043953,0.159890,0.962804,0.907355,0.962860


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



✅ DISTILBERT TRAINING COMPLETE!
⏱️  Training time: 218.64 minutes


In [ ]:
# MODEL 1: Evaluate DistilBERT
distilbert_results = evaluate_model(
    distilbert_trainer,
    distilbert_test_dataset,
    label_encoder,
    "Test"
)

# Save evaluation results
distilbert_results['model_name'] = 'DistilBERT'
distilbert_results['training_time_minutes'] = training_time / 60


EVALUATING ON TEST SET



📊 Overall Metrics:
   Accuracy:    0.9615 (96.15%)
   F1 Macro:    0.8955
   F1 Weighted: 0.9616

📋 Per-Class Performance:
------------------------------------------------------------
              precision    recall  f1-score   support

          A1     0.5000    0.6667    0.5714        12
          A2     0.9833    0.9755    0.9793      5907
          B1     0.9561    0.9616    0.9589      5914
          B2     0.9556    0.9574    0.9565      5961
          C1     0.9442    0.9442    0.9442      3173
          C2     0.9632    0.9620    0.9626      1553

    accuracy                         0.9615     22520
   macro avg     0.8837    0.9112    0.8955     22520
weighted avg     0.9617    0.9615    0.9616     22520


🔢 Confusion Matrix:
------------------------------------------------------------
    A1    A2    B1    B2    C1    C2
A1   8     0     4     0     0     0
A2   6  5762   124    11     3     1
B1   2    87  5687   128     7     3
B2   0     7   127  5707   111     9
C1   

In [ ]:
# MODEL 1: Save DistilBERT Model
print("\n" + "="*60)
print("SAVING DISTILBERT MODEL")
print("="*60)

distilbert_save_path = "./distilbert_cefr_classifier"
distilbert_trainer.save_model(distilbert_save_path)
distilbert_tokenizer.save_pretrained(distilbert_save_path)

print(f"\n✅ DistilBERT model saved to: {distilbert_save_path}")
print("="*60)


SAVING DISTILBERT MODEL


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ DistilBERT model saved to: ./distilbert_cefr_classifier


---
# ⚡ MODEL 2: ELECTRA

Efficient training with discriminative approach.
- **Model**: `google/electra-base-discriminator`
- **Parameters**: 110M
- **Training Time**: ~40-60 minutes on GPU

---

In [ ]:
# MODEL 2: Load ELECTRA Model and Tokenizer
print("="*60)
print("MODEL 2: ELECTRA - LOADING")
print("="*60)

electra_model_name = "google/electra-base-discriminator"

print(f"\nLoading tokenizer: {electra_model_name}")
electra_tokenizer = AutoTokenizer.from_pretrained(electra_model_name)

print(f"Loading model with {num_labels} labels")
electra_model = AutoModelForSequenceClassification.from_pretrained(
    electra_model_name,
    num_labels=num_labels
).to(device)

print(f"\n✅ ELECTRA loaded successfully!")
print(f"   Parameters: {sum(p.numel() for p in electra_model.parameters()) / 1e6:.1f}M")
print("="*60)

MODEL 2: ELECTRA - LOADING

Loading tokenizer: google/electra-base-discriminator


config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: google/electra-base-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
electra.embeddings_project.weight                 | UNEXPECTED | 
electra.embeddings_project.bias                   | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- 

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]


✅ ELECTRA loaded successfully!
   Parameters: 109.5M


In [ ]:
# MODEL 2: Prepare ELECTRA Datasets
print("="*60)
print("PREPARING ELECTRA DATASETS")
print("="*60)

electra_train_dataset = CEFRDataset(X_train, y_train, electra_tokenizer, max_length=512)
electra_val_dataset = CEFRDataset(X_val, y_val, electra_tokenizer, max_length=512)
electra_test_dataset = CEFRDataset(X_test, y_test, electra_tokenizer, max_length=512)

print(f"\n✅ Datasets created:")
print(f"   Training:   {len(electra_train_dataset)} samples")
print(f"   Validation: {len(electra_val_dataset)} samples")
print(f"   Test:       {len(electra_test_dataset)} samples")
print("="*60)

PREPARING ELECTRA DATASETS

✅ Datasets created:
   Training:   108468 samples
   Validation: 19142 samples
   Test:       22520 samples


In [ ]:
# MODEL 2: Configure ELECTRA Training
print("="*60)
print("CONFIGURING ELECTRA TRAINING")
print("="*60)

# Training arguments
electra_training_args = TrainingArguments(
    output_dir='./electra_model',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=3e-5,  # Slightly higher LR for ELECTRA
    weight_decay=0.01,
    warmup_steps=500,
    logging_dir='./logs/electra',
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    save_total_limit=2,
    seed=RANDOM_STATE,
    fp16=torch.cuda.is_available(),
)

# Initialize Trainer with class weights
electra_trainer = WeightedTrainer(
    model=electra_model,
    args=electra_training_args,
    train_dataset=electra_train_dataset,
    eval_dataset=electra_val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("\n✅ ELECTRA Trainer configured!")
print(f"   Epochs: {electra_training_args.num_train_epochs}")
print(f"   Batch Size: {electra_training_args.per_device_train_batch_size}")
print(f"   Learning Rate: {electra_training_args.learning_rate}")
print(f"   Class Weights: Applied ✓")
print("="*60)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


CONFIGURING ELECTRA TRAINING

✅ ELECTRA Trainer configured!
   Epochs: 4
   Batch Size: 16
   Learning Rate: 3e-05
   Class Weights: Applied ✓


In [ ]:
# MODEL 2: Train ELECTRA
print("="*60)
print("TRAINING ELECTRA MODEL")
print("="*60)
print("\n🚀 Starting training... This may take 40-60 minutes on GPU\n")

start_time_electra = time.time()

# Train
electra_trainer.train()

training_time_electra = time.time() - start_time_electra

print("\n" + "="*60)
print("✅ ELECTRA TRAINING COMPLETE!")
print("="*60)
print(f"⏱️  Training time: {training_time_electra/60:.2f} minutes")
print("="*60)

TRAINING ELECTRA MODEL

🚀 Starting training... This may take 40-60 minutes on GPU



Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,0.141268,0.178464,0.951625,0.897074,0.951766
2,0.129288,0.146541,0.958834,0.890634,0.958940
3,0.062297,0.149157,0.961342,0.917377,0.961363
4,0.037134,0.177628,0.961237,0.911895,0.961248


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.LayerNorm.weight', 'electra.encoder.layer.2.attention.output.LayerNorm.bias', 'electra.encoder.layer.2.output.LayerNorm.weight', 'electra.encoder.layer.2.output.LayerNorm.bias', 'electra.encoder.layer.3.attention.output.LayerNorm.weight', 'electra.encoder.layer.3.attention.output.LayerNorm.bias', 'electra.encoder.layer.3.output.LayerNorm.weight', 'electra.encoder.layer.3.output.Laye


✅ ELECTRA TRAINING COMPLETE!
⏱️  Training time: 427.33 minutes


In [ ]:
# MODEL 2: Evaluate ELECTRA
electra_results = evaluate_model(
    electra_trainer,
    electra_test_dataset,
    label_encoder,
    "Test"
)

# Save evaluation results
electra_results['model_name'] = 'ELECTRA'
electra_results['training_time_minutes'] = training_time_electra / 60


EVALUATING ON TEST SET



📊 Overall Metrics:
   Accuracy:    0.9608 (96.08%)
   F1 Macro:    0.9017
   F1 Weighted: 0.9608

📋 Per-Class Performance:
------------------------------------------------------------
              precision    recall  f1-score   support

          A1     0.6364    0.5833    0.6087        12
          A2     0.9790    0.9792    0.9791      5907
          B1     0.9554    0.9599    0.9577      5914
          B2     0.9601    0.9485    0.9543      5961
          C1     0.9322    0.9540    0.9430      3173
          C2     0.9770    0.9581    0.9675      1553

    accuracy                         0.9608     22520
   macro avg     0.9067    0.8972    0.9017     22520
weighted avg     0.9609    0.9608    0.9608     22520


🔢 Confusion Matrix:
------------------------------------------------------------
    A1    A2    B1    B2    C1    C2
A1   7     1     4     0     0     0
A2   4  5784   101    12     6     0
B1   0   117  5677   114     6     0
B2   0     3   157  5654   145     2
C1   

In [ ]:
# MODEL 2: Save ELECTRA Model
print("\n" + "="*60)
print("SAVING ELECTRA MODEL")
print("="*60)

electra_save_path = "./electra_cefr_classifier"
electra_trainer.save_model(electra_save_path)
electra_tokenizer.save_pretrained(electra_save_path)

print(f"\n✅ ELECTRA model saved to: {electra_save_path}")
print("="*60)


SAVING ELECTRA MODEL


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ ELECTRA model saved to: ./electra_cefr_classifier


---
# 📊 MODEL COMPARISON

Comparing DistilBERT vs ELECTRA performance

---

In [ ]:
# Compare Models
print("="*80)
print(" " * 25 + "MODEL COMPARISON")
print("="*80)

comparison_df = pd.DataFrame({
    'Model': ['DistilBERT', 'ELECTRA'],
    'Accuracy (%)': [
        distilbert_results['accuracy'] * 100,
        electra_results['accuracy'] * 100
    ],
    'F1 Macro': [
        distilbert_results['f1_macro'],
        electra_results['f1_macro']
    ],
    'F1 Weighted': [
        distilbert_results['f1_weighted'],
        electra_results['f1_weighted']
    ],
    'Training Time (min)': [
        distilbert_results['training_time_minutes'],
        electra_results['training_time_minutes']
    ]
})

print("\n" + comparison_df.to_string(index=False))

# Determine winner
print("\n" + "="*80)
if distilbert_results['accuracy'] > electra_results['accuracy']:
    winner = 'DistilBERT'
    diff = (distilbert_results['accuracy'] - electra_results['accuracy']) * 100
else:
    winner = 'ELECTRA'
    diff = (electra_results['accuracy'] - distilbert_results['accuracy']) * 100

print(f"🏆 WINNER: {winner}")
print(f"   Accuracy difference: {diff:.2f}%")

if distilbert_results['training_time_minutes'] < electra_results['training_time_minutes']:
    faster = 'DistilBERT'
    time_diff = electra_results['training_time_minutes'] - distilbert_results['training_time_minutes']
else:
    faster = 'ELECTRA'
    time_diff = distilbert_results['training_time_minutes'] - electra_results['training_time_minutes']

print(f"\n⚡ FASTER MODEL: {faster}")
print(f"   Time saved: {time_diff:.2f} minutes")

print("="*80)

                         MODEL COMPARISON

     Model  Accuracy (%)  F1 Macro  F1 Weighted  Training Time (min)
DistilBERT     96.154529  0.895498     0.961602           218.644371
   ELECTRA     96.079041  0.901699     0.960807           427.328854

🏆 WINNER: DistilBERT
   Accuracy difference: 0.08%

⚡ FASTER MODEL: DistilBERT
   Time saved: 208.68 minutes


In [ ]:
# Inference Function for New Paragraphs
def predict_cefr_level(text, model_choice='distilbert'):
    """
    Predict CEFR level for a new paragraph

    Args:
        text (str): The input paragraph
        model_choice (str): 'distilbert' or 'electra'

    Returns:
        dict: Prediction results with label and confidence
    """
    # Load model and tokenizer
    if model_choice.lower() == 'distilbert':
        model_path = "./distilbert_cefr_classifier"
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
    else:
        model_path = "./electra_cefr_classifier"
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)

    # Tokenize
    inputs = tokenizer(
        text,
        add_special_tokens=True,
        max_length=512,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    ).to(device)

    # Predict
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1)
        predicted_class = torch.argmax(probabilities, dim=1).item()
        confidence = probabilities[0][predicted_class].item()

    # Get label name
    predicted_label = label_encoder.classes_[predicted_class]

    # Get all probabilities
    all_probs = {
        label_encoder.classes_[i]: probabilities[0][i].item()
        for i in range(len(label_encoder.classes_))
    }

    return {
        'predicted_level': predicted_label,
        'confidence': confidence,
        'all_probabilities': all_probs
    }

print("✅ Inference function created!")
print("\nUsage:")
print("  result = predict_cefr_level('Your paragraph here', model_choice='distilbert')")
print("  print(result['predicted_level'])")

✅ Inference function created!

Usage:
  result = predict_cefr_level('Your paragraph here', model_choice='distilbert')
  print(result['predicted_level'])


In [ ]:
# Test Inference with Examples
print("="*80)
print("TESTING INFERENCE FUNCTION")
print("="*80)

# Test examples from different CEFR levels
test_examples = [
    "I like cats. They are nice. I have one cat.",  # Simple sentence (A1-A2)
    "Yesterday I went to the store and bought some groceries. It was a pleasant experience.",  # Intermediate (B1)
    "The implementation of sustainable practices in modern corporations requires a comprehensive understanding of both environmental and economic factors.",  # Advanced (C1-C2)
]

print("\n" + "-"*80)
for i, example in enumerate(test_examples, 1):
    print(f"\nExample {i}: {example[:80]}...")

    # Predict with both models
    distilbert_pred = predict_cefr_level(example, 'distilbert')
    electra_pred = predict_cefr_level(example, 'electra')

    print(f"\n  DistilBERT: {distilbert_pred['predicted_level']} (confidence: {distilbert_pred['confidence']:.3f})")
    print(f"  ELECTRA:    {electra_pred['predicted_level']} (confidence: {electra_pred['confidence']:.3f})")
    print("-"*80)

print("\n✅ Models are ready for inference!")
print("="*80)

TESTING INFERENCE FUNCTION

--------------------------------------------------------------------------------

Example 1: I like cats. They are nice. I have one cat....


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


  DistilBERT: A2 (confidence: 0.693)
  ELECTRA:    A1 (confidence: 0.786)
--------------------------------------------------------------------------------

Example 2: Yesterday I went to the store and bought some groceries. It was a pleasant exper...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


  DistilBERT: A2 (confidence: 0.946)
  ELECTRA:    A2 (confidence: 0.859)
--------------------------------------------------------------------------------

Example 3: The implementation of sustainable practices in modern corporations requires a co...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


  DistilBERT: B2 (confidence: 0.947)
  ELECTRA:    B2 (confidence: 0.912)
--------------------------------------------------------------------------------

✅ Models are ready for inference!


---
# ✅ TRAINING COMPLETE!

Both models have been trained and evaluated:
- **DistilBERT**: Fast and efficient
- **ELECTRA**: Efficient training approach

## 📁 Saved Files:
- `./distilbert_cefr_classifier/` - DistilBERT model
- `./electra_cefr_classifier/` - ELECTRA model
- `label_encoder.pkl` - Label encoder
- `class_weights.pkl` - Class weights

## 🚀 Next Steps:
1. Compare the results above
2. Choose the best performing model
3. Use `predict_cefr_level()` function for new paragraphs
4. Deploy the model for production use

---